In [ ]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .config("spark.driver.extraJavaOptions", "-Djava.library.path=C:/tools/hadoop/bin") \
    .getOrCreate()

In [3]:
df_green = spark.read.parquet('../data/pq/green/*/*')

In [9]:
# df_green.show()
# df_green.rdd
# df_green.limit(5).rdd.collect()
# df_green.take(5)

In [10]:
"""
SELECT
    date_trunc('hour', lpep_pickup_datetime) AS hour,
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    count(1) as number_records
FROM
    green
where
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
order by
    1, 2
"""

"\nSELECT\n    date_trunc('hour', lpep_pickup_datetime) AS hour,\n    PULocationID AS zone,\n\n    SUM(total_amount) AS amount,\n    count(1) as number_records\nFROM\n    green\nwhere\n    lpep_pickup_datetime >= '2020-01-01 00:00:00'\nGROUP BY\n    1, 2\norder by\n    1, 2\n"

In [22]:
rdd = df_green \
    .select('lpep_pickup_datetime', 'PULocationID', 'total_amount') \
    .limit(1000) \
    .rdd

In [26]:
from datetime import datetime

In [32]:
start = datetime(year=2020, month=1, day=1)

def filter_outliers(row):
    return row.lpep_pickup_datetime >= start

In [35]:
rows = rdd.take(10)
row = rows[0]

In [39]:
# row.lpep_pickup_datetime.replace(minute=0, second=0, microsecond=0)
row

Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 2, 19, 57, 21), PULocationID=82, total_amount=9.36)

In [40]:
def prepare_for_grouping(row):
    hour = row.lpep_pickup_datetime.replace(minute=0, second=0, microsecond=0)
    zone = row.PULocationID
    key = (hour, zone)
    amount = row.total_amount
    count = 1
    value = (amount, count)

    return (key, value)

In [41]:
# rdd.filter(lambda row: True).take(1)
# rdd.filter(lambda row: False).take(1)
# rdd.filter(filter_outliers).take(1)

In [43]:
def calculate_revenue(left_value, right_value):
    left_amount, left_count = left_value
    right_amount, right_count = right_value

    output_amount = left_amount + right_amount
    output_count = left_count + right_count
    
    return (output_amount, output_count)

In [49]:
from collections import namedtuple

In [52]:
RevenueRow = namedtuple('RevenueRow', ['hour', 'zone', 'revenue', 'count'])

In [53]:
def unwrap(row):
    return RevenueRow(
        hour=row[0][0],
        zone=row[0][1],
        revenue=row[1][0],
        count=row[1][1]
    )

In [59]:
from pyspark.sql import types

result_schema = types.StructType([
    types.StructField('hour', types.TimestampType(), True),
    types.StructField('zone', types.IntegerType(), True),
    types.StructField('revenue', types.DoubleType(), True),
    types.StructField('count', types.IntegerType(), True)
])

In [60]:
df_result = rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .map(unwrap) \
    .toDF(result_schema)

In [63]:
# df_result.schema
# df_result.show()

In [64]:
df_result.write.parquet('../tmp/green-revenue')

In [69]:
columns = ['VendorID', 'lpep_pickup_datetime', 'PULocationID', 'DOLocationID', 'trip_distance']

duration_rdd = df_green \
    .select(columns) \
    .limit(1000) \
    .rdd

In [70]:
# duration_rdd.take(5)

[Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 2, 19, 57, 21), PULocationID=82, DOLocationID=160, trip_distance=1.17),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 2, 17, 0, 42), PULocationID=7, DOLocationID=223, trip_distance=2.28),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 4, 20, 55, 5), PULocationID=7, DOLocationID=129, trip_distance=2.44),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 2, 18, 50, 49), PULocationID=75, DOLocationID=263, trip_distance=1.04),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 4, 0, 10, 39), PULocationID=37, DOLocationID=189, trip_distance=4.89)]

In [77]:
import pandas as pd

In [78]:
rows = duration_rdd.take(10)

In [88]:
df = pd.DataFrame(rows, columns=columns)

In [89]:
list(df.itertuples())

[Pandas(Index=0, VendorID=2, lpep_pickup_datetime=Timestamp('2020-01-02 19:57:21'), PULocationID=82, DOLocationID=160, trip_distance=1.17),
 Pandas(Index=1, VendorID=2, lpep_pickup_datetime=Timestamp('2020-01-02 17:00:42'), PULocationID=7, DOLocationID=223, trip_distance=2.28),
 Pandas(Index=2, VendorID=2, lpep_pickup_datetime=Timestamp('2020-01-04 20:55:05'), PULocationID=7, DOLocationID=129, trip_distance=2.44),
 Pandas(Index=3, VendorID=2, lpep_pickup_datetime=Timestamp('2020-01-02 18:50:49'), PULocationID=75, DOLocationID=263, trip_distance=1.04),
 Pandas(Index=4, VendorID=2, lpep_pickup_datetime=Timestamp('2020-01-04 00:10:39'), PULocationID=37, DOLocationID=189, trip_distance=4.89),
 Pandas(Index=5, VendorID=2, lpep_pickup_datetime=Timestamp('2020-01-01 12:29:31'), PULocationID=7, DOLocationID=165, trip_distance=13.75),
 Pandas(Index=6, VendorID=2, lpep_pickup_datetime=Timestamp('2020-01-03 19:10:17'), PULocationID=226, DOLocationID=226, trip_distance=0.31),
 Pandas(Index=7, Vend

In [80]:
pd.DataFrame(rows, columns=columns)

,VendorID,lpep_pickup_datetime,PULocationID,DOLocationID,trip_distance
0,2,2020-01-02 19:57:21,82,160,1.17
1,2,2020-01-02 17:00:42,7,223,2.28
2,2,2020-01-04 20:55:05,7,129,2.44
3,2,2020-01-02 18:50:49,75,263,1.04
4,2,2020-01-04 00:10:39,37,189,4.89
5,2,2020-01-01 12:29:31,7,165,13.75
6,2,2020-01-03 19:10:17,226,226,0.31
7,2,2020-01-04 18:08:16,7,226,1.23
8,2,2020-01-04 19:19:34,130,216,2.40
9,1,2020-01-03 19:40:59,41,263,2.30


In [84]:
# model = ...

def model_predict(df):
#    y_pred = model.predict(df)
    y_pred = df.trip_distance * 5
    return y_pred

In [100]:
def infinite_seq():
    i = 0

    while True:
        yield i
        i = i + 1

        if i > 15:
            break

In [101]:
seq = infinite_seq()

In [102]:
# next(seq)
list(seq)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

In [97]:
for i in seq:
    print(i)

    if i >= 10:
        break

3
4
5
6
7
8
9
10


In [105]:
def apply_model_in_batch(rows):
    df = pd.DataFrame(rows, columns=columns)
    predictions = model_predict(df)
    df['predicted_duration'] = predictions

    for row in df.itertuples(index=False):
        yield row

In [107]:
df_predicts = duration_rdd \
    .mapPartitions(apply_model_in_batch) \
    .toDF() \
    .drop('Index')

In [109]:
df_predicts.select('predicted_duration').show()

+------------------+
|predicted_duration|
+------------------+
|              5.85|
|11.399999999999999|
|              12.2|
|               5.2|
|             24.45|
|             68.75|
|              1.55|
|              6.15|
|              12.0|
|              11.5|
|               7.0|
|20.950000000000003|
|               3.0|
|               6.5|
|             13.65|
|              6.45|
|              10.5|
|               6.2|
|              11.5|
|              14.5|
+------------------+
only showing top 20 rows
